# Wind rose data

Here we demonstrate how mean results over wind rose data are calculated in `foxes`. We need the following imports:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import foxes
import foxes.variables as FV

First, we create the engine:

In [ ]:
engine = foxes.Engine.new(
    "process", chunk_size_states=1000, chunk_size_points=3000, n_procs=4
)

Next, let's setup the wind farm. We choose 6 x 5 turbines on a regular grid:

In [ ]:
farm = foxes.WindFarm()
foxes.input.farm_layout.add_grid(
    farm=farm,
    xy_base=[0.0, 0.0],
    step_vectors=[[900.0, 50.0], [-80.0, 500.0]],
    steps=[6, 5],
    turbine_models=["NREL5MW", "kTI_k1_05"],
    verbosity=0,
)

ax = foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure", parameters=dict(figsize=(4, 3))
    )
)
plt.show()

Next, we create the states based on the static data file `wind_rose_bremen.csv.gz`. The data represents a (coarse) wind rose with 216 states. Each of them consists of the wind direction and wind speed bin centres, and the respective statistical weight of the bin (normalized such that 1 represents 100%):
```
state,wd,ws,weight
0,0.0,3.5,0.00158
1,0.0,6.0,0.00244
2,0.0,8.5,0.00319
3,0.0,12.5,0.0036700002
4,0.0,17.5,0.00042
...
```
Let's create the states object and have a look at the wind rose:

In [ ]:
states = foxes.input.states.StatesTable(
    data_source="wind_rose_bremen.csv",
    output_vars=[FV.WS, FV.WD, FV.TI, FV.RHO],
    var2col={FV.WS: "ws", FV.WD: "wd", FV.WEIGHT: "weight"},
    fixed_vars={FV.RHO: 1.225, FV.TI: 0.05},
)

with engine:
    rose_farm = foxes.WindFarm()
    rose_farm.add_turbine(
        foxes.Turbine([0.0, 0.0], H=100.0, D=1.0, turbine_models=["null_type"]),
        verbosity=0,
    )
    foxes.algorithms.Downwind(
        rose_farm, states, wake_models=[], verbosity=0
    ).calc_outputs(
        outputs=foxes.models.outputs.StatesRosePlotOutput(
            point=[0.0, 0.0, 100.0], plot=True, plot_pars=dict(figsize=(6, 6))
        )
    )[0]
plt.show()

Use `Bastankhah2014_linear` for wind-speed deficits and `CrespoHernandez_max` for turbulence intensity. These use linear and maximum superposition, respectively. Both use `k = 0.5 * TI`, as configured by the farm's turbine models. `partial_wakes=None` selects their default partial-wake models:

In [ ]:
algo = foxes.algorithms.Downwind(
    farm,
    states,
    rotor_model="centre",
    wake_models=["Bastankhah2014_linear", "CrespoHernandez_max"],
    partial_wakes=None,
    verbosity=0,
)

We can now calculate the results:

In [ ]:
algo.mbook.outputs["farm_results"].fdata_variables.extend([FV.EFF, FV.K, FV.CT])

with engine:
    farm_results = algo.calc_farm()


fig, axs = plt.subplots(2, 1, figsize=(6, 7))

foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure",
        parameters=dict(
            fig=fig,
            ax=axs[0],
            color_by="mean_REWS",
            title="Mean REWS [m/s]",
            s=150,
            annotate=0,
        ),
    ),
    farm_results=farm_results,
)
foxes.algorithms.Downwind(
    farm,
    foxes.input.states.SingleStateStates(ws=8, wd=270, ti=0.08, rho=1.225),
    wake_models=[],
    verbosity=0,
).calc_outputs(
    outputs=foxes.models.outputs.FarmLayoutOutput(
        operation="get_figure",
        parameters=dict(
            fig=fig,
            ax=axs[1],
            color_by="mean_EFF",
            title="Mean efficiency [%]",
            s=150,
            annotate=0,
        ),
    ),
    farm_results=farm_results,
)
plt.show()

Finally, we display the mean wind speed field as a 2D plot, including wake effects:

In [ ]:
with engine:
    plot_data = algo.calc_outputs(
        outputs=foxes.models.outputs.FlowPlots2D(
            plane="xy",
            reduction="mean",
            plot=False,
            data_format="numpy",
            var=FV.WS,
            grid_pars=dict(resolution=30),
        ),
        farm_results=farm_results,
    )
algo.calc_outputs(
    outputs=foxes.models.outputs.FlowPlots2D(
        plane="xy",
        reduction="mean",
        plot=True,
        data_format="numpy",
        var=FV.WS,
        grid_pars=dict(resolution=30),
    ),
    farm_results=farm_results,
)
plt.show()